In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

DATA_DIR = Path(r"C:\\Users\\ahzam\Desktop\\Projects\\JetGuard\\dataset")

columns = (
    ["engine_id", "cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train_df = pd.read_csv(
    DATA_DIR / "train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

test_df = pd.read_csv(
    DATA_DIR / "test_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

rul_df = pd.read_csv(
    DATA_DIR / "RUL_FD001.txt",
    sep=r"\s+",
    header=None,
    names=["RUL"]
)

In [2]:
print("Missing values:")
print(train_df.isnull().sum().sum())

print("Duplicate rows:")
print(train_df.duplicated().sum())

print("Infinite values:")
print(np.isinf(train_df.select_dtypes("number")).sum().sum())

Missing values:
0
Duplicate rows:
0
Infinite values:
0


In [3]:
constant_sensors = [
    "sensor_1",
    "sensor_5",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19"
]

In [4]:
print(train_df[constant_sensors].nunique())

sensor_1     1
sensor_5     1
sensor_10    1
sensor_16    1
sensor_18    1
sensor_19    1
dtype: int64


In [5]:
train_df["max_cycle"] = (
    train_df.groupby("engine_id")["cycle"]
    .transform("max")
)

train_df["RUL"] = (
    train_df["max_cycle"] - train_df["cycle"]
)

train_df.drop(
    columns=["max_cycle"],
    inplace=True
)

In [6]:
train_df[
    ["engine_id", "cycle", "RUL"]
].head(15)

,engine_id,cycle,RUL
0,1,1,191
1,1,2,190
2,1,3,189
3,1,4,188
4,1,5,187
5,1,6,186
6,1,7,185
7,1,8,184
8,1,9,183
9,1,10,182


In [7]:
print(
    train_df.groupby("engine_id")["RUL"].min().unique()
)

[0]


In [8]:
RUL_CAP = 125

train_df["RUL_capped"] = (
    train_df["RUL"].clip(upper=RUL_CAP)
)

In [9]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(
    output_dir / "train_FD001_processed.csv",
    index=False
)